# Validación de staging: superficie centinela, precios extremos y tipos de dato

**Objetivo:** documentar los análisis y validaciones realizados **después** de construir
`models/staging/stg_toctoc.sql` y `models/staging/stg_portal_inmobiliario.sql` — hallazgos que
no habían sido identificados en el perfilado inicial (`01_exploration_toctoc.ipynb` /
`02_exploration_portal_inmobiliario.ipynb`), la evidencia usada para decidir su tratamiento, y
las validaciones que confirman que staging lo implementó correctamente.

Este notebook es solo de validación/diagnóstico: no transforma datos ni reemplaza la lógica de
negocio, que vive en `models/staging/`. Consulta directamente `raw` y `staging` en BigQuery.

**Nota (actualizado tras el diseño de marts)**: la investigación de este notebook trató inicialmente 1, 2 y 5 como un solo grupo. La clasificación final (ver `04_validate_marts.ipynb`) separó `1`/`2` como centinela (se anula `superficie_m2`) de `5`/`8`/`10`/`12` como "habitación" (superficie real, se conserva) — staging usa hoy la columna `superficie_categoria` en vez del booleano `superficie_es_centinela` que aparece en las celdas de investigación original de este notebook.

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery

load_dotenv("../.env")

# GOOGLE_APPLICATION_CREDENTIALS en .env es relativo a la raiz del proyecto; el notebook
# se ejecuta con cwd=notebooks/, asi que se resuelve explicitamente antes de crear el cliente.
cred_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
if cred_path and not os.path.isabs(cred_path):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str((Path("..") / cred_path).resolve())

PROJECT = os.getenv("GCP_PROJECT_ID")
DATASET_RAW = os.getenv("BQ_DATASET_RAW")
DATASET_STAGING = os.getenv("BQ_DATASET_STAGING")

bq = bigquery.Client(project=PROJECT)


def query(sql: str) -> pd.DataFrame:
    return bq.query(sql).to_dataframe()


VALORES_SOSPECHOSOS = [1, 2, 5, 250, 400, 650, 999, 1200]
FUENTES = [("toctoc", "superficie_m2"), ("portal_inmobiliario", "superficie_total_m2")]

## 1. `superficie_m2`: impacto de los valores sospechosos

Durante la revisión manual de `staging` se detectaron picos de `superficie_m2` en valores
exactos (1, 2, 5, 250, 400, 650, 999, 1200) que no se habían marcado como problema en el
perfilado inicial. Esta sección cuantifica su impacto en `raw`, antes de cualquier tratamiento.

In [2]:
for tabla, col in FUENTES:
    total = query(f"SELECT COUNT(*) AS n FROM `{PROJECT}.{DATASET_RAW}.{tabla}`")["n"][0]
    valores_str = ",".join(str(v) for v in VALORES_SOSPECHOSOS)
    df = query(f'''
        SELECT CAST({col} AS FLOAT64) AS superficie, COUNT(*) AS n
        FROM `{PROJECT}.{DATASET_RAW}.{tabla}`
        WHERE CAST({col} AS FLOAT64) IN ({valores_str})
        GROUP BY superficie ORDER BY superficie
    ''')
    df["pct_del_total"] = (df["n"] / total * 100).round(3)
    print(f"--- {tabla} (total filas = {total}) ---")
    display(df)
    print(f"Total centinela: {df['n'].sum()} ({df['n'].sum() / total * 100:.2f}% del total)\n")

C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


--- toctoc (total filas = 124374) ---


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,superficie,n,pct_del_total
0,1.0,107,0.086
1,2.0,134,0.108
2,5.0,125,0.101
3,250.0,148,0.119
4,400.0,149,0.12
5,650.0,164,0.132
6,999.0,150,0.121
7,1200.0,146,0.117


Total centinela: 1123 (0.90% del total)



C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


--- portal_inmobiliario (total filas = 122273) ---


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,superficie,n,pct_del_total
0,1.0,123,0.101
1,2.0,94,0.077
2,5.0,129,0.106
3,250.0,141,0.115
4,400.0,132,0.108
5,650.0,147,0.12
6,999.0,162,0.132
7,1200.0,141,0.115


Total centinela: 1069 (0.87% del total)



### 1.1 Distribución mensual (¿es estable en el tiempo?)

In [3]:
for tabla, col in FUENTES:
    valores_str = ",".join(str(v) for v in VALORES_SOSPECHOSOS)
    df = query(f'''
        WITH base AS (
          SELECT
            FORMAT_DATE('%Y-%m', DATE(SUBSTR(fecha_scraping, 1, 10))) AS mes,
            CAST({col} AS FLOAT64) AS superficie
          FROM `{PROJECT}.{DATASET_RAW}.{tabla}`
        ),
        por_mes AS (
          SELECT mes, COUNT(*) AS n_total_mes FROM base GROUP BY mes
        )
        SELECT b.mes, COUNT(*) AS n_centinela, ANY_VALUE(p.n_total_mes) AS n_total_mes
        FROM base b JOIN por_mes p ON b.mes = p.mes
        WHERE b.superficie IN ({valores_str})
        GROUP BY b.mes ORDER BY b.mes
    ''')
    df["pct"] = (df["n_centinela"] / df["n_total_mes"] * 100).round(2)
    print(f"--- {tabla}: rango de % mensual = {df['pct'].min()}% - {df['pct'].max()}% "
          f"(sin tendencia ni mes atípico) ---")

C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


--- toctoc: rango de % mensual = 0.65% - 1.14% (sin tendencia ni mes atípico) ---


--- portal_inmobiliario: rango de % mensual = 0.63% - 1.15% (sin tendencia ni mes atípico) ---


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


### 1.2 ¿Diferencias relevantes entre fuentes?

No. Magnitud similar (Toctoc 0.90%, Portal 0.87%), mismos 8 valores presentes en ambas, y la
misma estabilidad mensual en las dos — es el mismo fenómeno en ambas fuentes, no una diferencia
entre ellas.

## 2. Evidencia utilizada para considerarlos centinela

Un valor de superficie "sospechoso" se trató como centinela cuando cumple, en ambas fuentes:

1. Es un pico exacto **sin ningún valor vecino** (ninguna fila con superficie entre `v-2` y
   `v+2`, salvo el valor exacto) — una superficie real medida no se comporta así, ya que el
   resto de la distribución tiene valores decimales finos (44.3, 42.8, etc.).
2. Aparece **repartido parejo en los 24 meses**, sin concentrarse en una corrida puntual.
3. Ocurre siempre en `tipo_propiedad = departamento`.

In [4]:
for tabla, col in FUENTES:
    print(f"--- {tabla}: vecinos alrededor de cada valor sospechoso ---")
    for v in VALORES_SOSPECHOSOS:
        rango = 0.5 if v <= 5 else 2
        df = query(f'''
            SELECT
              COUNTIF(CAST({col} AS FLOAT64) = {v}) AS n_exacto,
              COUNTIF(CAST({col} AS FLOAT64) BETWEEN {v - rango} AND {v + rango}
                      AND CAST({col} AS FLOAT64) != {v}) AS n_vecinos
            FROM `{PROJECT}.{DATASET_RAW}.{tabla}`
        ''')
        print(f"  {v}: n_exacto={df['n_exacto'][0]}, n_vecinos={df['n_vecinos'][0]}")
    print()

--- toctoc: vecinos alrededor de cada valor sospechoso ---


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  1: n_exacto=107, n_vecinos=0


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  2: n_exacto=134, n_vecinos=0


...

  650: n_exacto=147, n_vecinos=0


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  999: n_exacto=162, n_vecinos=0


  1200: n_exacto=141, n_vecinos=0



C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## 3. Relación con `precio_m2`

La evidencia de correlación con precio extremo **no es igual para los 8 valores** — se agrupan
en 3 comportamientos distintos.

### 3.1 `999` y `1200` — evidencia más fuerte

La mayoría de estas filas también tiene un precio extremo en términos absolutos.

In [5]:
for tabla, col, precio_expr in [
    ("toctoc", "superficie_m2",
     "(divisa='UF' AND CAST(precio AS FLOAT64) > 100) OR (divisa='CLP' AND CAST(precio AS FLOAT64) > 5000000)"),
    ("portal_inmobiliario", "superficie_total_m2",
     "CAST(precio_clp AS FLOAT64) > 5000000 OR CAST(precio_uf AS FLOAT64) > 100"),
]:
    df = query(f'''
        SELECT
          COUNTIF({precio_expr}) AS n_precio_extremo,
          COUNT(*) AS n_total
        FROM `{PROJECT}.{DATASET_RAW}.{tabla}`
        WHERE CAST({col} AS FLOAT64) IN (999, 1200)
    ''')
    pct = df["n_precio_extremo"][0] / df["n_total"][0] * 100
    print(f"{tabla}: {df['n_precio_extremo'][0]} de {df['n_total'][0]} filas centinela "
          f"999/1200 tienen precio extremo ({pct:.0f}%)")

C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


toctoc: 295 de 296 filas centinela 999/1200 tienen precio extremo (100%)


portal_inmobiliario: 303 de 303 filas centinela 999/1200 tienen precio extremo (100%)


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


### 3.2 `250`, `400` y `650` — misma firma estadística, comportamiento de precio distinto

Menor correlación con precio extremo (~31-39%, no ~56-99%), y el `precio_por_m2` resultante es
**más bajo** que la mediana general, no más alto — compatible con propiedades grandes reales
(economía de escala en el precio por m²). Se documentan como sospechosos por decisión
metodológica (comparten la firma de "pico aislado"), **no como error demostrado**.

In [6]:
df = query(f'''
    SELECT
      COUNTIF((divisa='UF' AND CAST(precio AS FLOAT64) > 100)
              OR (divisa='CLP' AND CAST(precio AS FLOAT64) > 5000000)) AS n_precio_extremo,
      COUNT(*) AS n_total
    FROM `{PROJECT}.{DATASET_RAW}.toctoc`
    WHERE CAST(superficie_m2 AS FLOAT64) IN (250, 400, 650)
''')
pct = df["n_precio_extremo"][0] / df["n_total"][0] * 100
print(f"toctoc: {df['n_precio_extremo'][0]} de {df['n_total'][0]} filas 250/400/650 "
      f"tienen precio extremo ({pct:.0f}%) — vs. 56-99% en 999/1200")

toctoc: 141 de 461 filas 250/400/650 tienen precio extremo (31%) — vs. 56-99% en 999/1200


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


### 3.3 `1`, `2` y `5` — origen directo de los `precio_por_m2` más extremos del dataset

31 de los 40 `precio_por_m2` más altos de Toctoc vienen de `superficie_m2 = 1`. El precio
**absoluto** de estas filas es normal (incluso más bajo que la mediana general) — la anomalía
está en la superficie, no en el precio.

In [7]:
df = query(f'''
    SELECT CAST(precio AS FLOAT64) / CAST(superficie_m2 AS FLOAT64) AS precio_por_m2,
           CAST(superficie_m2 AS FLOAT64) AS superficie_m2
    FROM `{PROJECT}.{DATASET_RAW}.toctoc`
    WHERE divisa = 'UF'
    ORDER BY precio_por_m2 DESC
    LIMIT 40
''')
print("Distribución de superficie_m2 entre los 40 precio_por_m2 (UF) más altos:")
print(df["superficie_m2"].value_counts())

Distribución de superficie_m2 entre los 40 precio_por_m2 (UF) más altos:
superficie_m2
1.0     32
2.0      4
5.0      3
10.0     1
Name: count, dtype: int64


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [8]:
for tabla, col, precio_col in [("toctoc", "superficie_m2", "precio"),
                                 ("portal_inmobiliario", "superficie_total_m2", "precio_clp")]:
    df = query(f'''
        SELECT
          COUNTIF(CAST({col} AS FLOAT64) <= 10) AS n_chica,
          AVG(IF(CAST({col} AS FLOAT64) <= 10, CAST({precio_col} AS FLOAT64), NULL)) AS precio_prom_chica,
          AVG(IF(CAST({col} AS FLOAT64) > 10, CAST({precio_col} AS FLOAT64), NULL)) AS precio_prom_resto
        FROM `{PROJECT}.{DATASET_RAW}.{tabla}`
        WHERE {precio_col} IS NOT NULL
    ''')
    print(f"{tabla}: superficie<=10 → precio promedio {df['precio_prom_chica'][0]:,.0f} "
          f"vs. resto {df['precio_prom_resto'][0]:,.0f} (no es más alto, es similar o menor)")

C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


toctoc: superficie<=10 → precio promedio 244,492 vs. resto 346,049 (no es más alto, es similar o menor)


portal_inmobiliario: superficie<=10 → precio promedio 503,746 vs. resto 775,983 (no es más alto, es similar o menor)


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## 4. Precios extremos y precio máximo de PortalInmobiliario

El precio máximo de PortalInmobiliario es \$141.035.000. Se revisa el registro completo y su
relación con superficie, comuna y tipo de propiedad antes de considerarlo anomalía.

In [9]:
df = query(f'''
    SELECT id, comuna, precio_clp, precio_uf, superficie_total_m2, tipo_propiedad,
           tipo_operacion, fecha_publicacion, fecha_scraping, contact_type
    FROM `{PROJECT}.{DATASET_RAW}.portal_inmobiliario`
    ORDER BY CAST(precio_clp AS FLOAT64) DESC
    LIMIT 5
''')
display(df)

C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,id,comuna,precio_clp,precio_uf,superficie_total_m2,tipo_propiedad,tipo_operacion,fecha_publicacion,fecha_scraping,contact_type
0,PI-2024-12-003280,Vitacura,141035000,3811.76,1200.0,DEPARTAMENTO,arriendo,2024-12-11 00:00:00.000000,2024-12-15 00:00:00.000000,agency
1,PI-2023-11-003442,Las Condes,103125000,2946.43,999.0,DEPARTAMENTO,ARRIENDO,2023-11-10 00:00:00.000000,2023-11-22 00:00:00.000000,owner_direct
2,PI-2023-07-003863,Santiago,88665000,2533.29,1200.0,Departamento,arriendo,2023-07-09 00:00:00.000000,2023-07-29 00:00:00.000000,agency
3,PI-2023-07-003077,Macul,78045000,2229.86,1200.0,departamento,arriendo,2023-05-05 00:00:00.000000,2023-07-15 00:00:00.000000,agency
4,PI-2023-07-001286,Macul,63485000,1813.86,999.0,Departamento,Arriendo,2023-03-25 00:00:00.000000,2023-07-08 00:00:00.000000,owner_direct


Los 5 precios más altos de PortalInmobiliario tienen `superficie_total_m2` en `{999, 1200}` —
el caso del precio máximo (id `PI-2024-12-003280`, Vitacura, superficie=1200) ya queda explicado
por el hallazgo de la sección 1-2: no es una anomalía de precio independiente, es consecuencia
de la superficie centinela asociada.

## 5. Casos extremos de superficie pequeña (1, 2, 5 m²)

Igual que con los valores altos, se revisó si estas filas tienen algo más en común (mes, tipo de
propiedad) además de la superficie.

In [10]:
for tabla, col in FUENTES:
    df = query(f'''
        SELECT tipo_propiedad, COUNT(*) AS n
        FROM `{PROJECT}.{DATASET_RAW}.{tabla}`
        WHERE CAST({col} AS FLOAT64) IN (1, 2, 5)
        GROUP BY tipo_propiedad
    ''')
    print(f"--- {tabla}: tipo_propiedad de filas con superficie en (1,2,5) ---")
    display(df)

--- toctoc: tipo_propiedad de filas con superficie en (1,2,5) ---


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,tipo_propiedad,n
0,departamento,114
1,Departamento,128
2,DEPARTAMENTO,124


--- portal_inmobiliario: tipo_propiedad de filas con superficie en (1,2,5) ---


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,tipo_propiedad,n
0,DEPARTAMENTO,118
1,Departamento,120
2,departamento,108


## 6. Otra anomalía revisada: granularidad de los duplicados exactos

Ya identificados en el perfilado inicial (`id` repetido). Durante la construcción de staging se
confirmó la causa: son copias literales dentro del mismo archivo mensual (nunca entre meses
distintos), y `id + fecha_scraping` no tiene ningún caso con contenido distinto entre las filas
de un mismo grupo — por eso `SELECT DISTINCT` es un tratamiento seguro en staging.

In [11]:
for tabla in ["toctoc", "portal_inmobiliario"]:
    df = query(f'''
        SELECT COUNT(*) AS total, COUNT(DISTINCT id) AS ids_distintos
        FROM `{PROJECT}.{DATASET_RAW}.{tabla}`
    ''')
    print(f"{tabla}: {df['total'][0]} filas, {df['ids_distintos'][0]} ids distintos, "
          f"diferencia (duplicados exactos) = {df['total'][0] - df['ids_distintos'][0]}")

C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


toctoc: 124374 filas, 124202 ids distintos, diferencia (duplicados exactos) = 172


portal_inmobiliario: 122273 filas, 122095 ids distintos, diferencia (duplicados exactos) = 178


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## 7. Validación: el precio se conserva aunque la superficie sea centinela

Se valida directamente sobre `staging` que ninguna fila perdió su precio por el tratamiento de
superficie, y que el caso del precio máximo de Portal quedó tratado correctamente.

In [12]:
for tabla in ["stg_toctoc", "stg_portal_inmobiliario"]:
    df = query(f'''
        SELECT
          COUNT(*) AS total,
          COUNTIF(superficie_categoria = 'centinela') AS n_centinela,
          COUNTIF(superficie_categoria = 'habitacion') AS n_habitacion,
          COUNTIF(superficie_categoria = 'centinela' AND superficie_m2 IS NOT NULL) AS centinela_con_m2_no_null_MAL,
          COUNTIF(superficie_categoria = 'habitacion' AND superficie_m2 IS NULL) AS habitacion_sin_m2_MAL,
          COUNTIF(superficie_categoria = 'centinela' AND precio_clp IS NULL) AS centinela_sin_precio_MAL
        FROM `{PROJECT}.{DATASET_STAGING}.{tabla}`
    ''')
    display(df)


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,total,n_centinela,n_habitacion,centinela_con_m2_no_null_MAL,habitacion_sin_m2_MAL,centinela_sin_precio_MAL
0,124202,996,475,0,0,0


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,total,n_centinela,n_habitacion,centinela_con_m2_no_null_MAL,habitacion_sin_m2_MAL,centinela_sin_precio_MAL
0,122095,939,490,0,0,0


In [13]:
df = query(f'''
    SELECT id, comuna, precio_clp, superficie_m2, superficie_m2_original, superficie_categoria
    FROM `{PROJECT}.{DATASET_STAGING}.stg_portal_inmobiliario`
    WHERE id = 'PI-2024-12-003280'
''')
display(df)


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,id,comuna,precio_clp,superficie_m2,superficie_m2_original,superficie_categoria
0,PI-2024-12-003280,Vitacura,141035000,NaN,1200.0,centinela


## 8. Validación: tipo de dato de `precio_clp`

In [14]:
df = query(f'''
    SELECT table_name, column_name, data_type
    FROM `{PROJECT}.{DATASET_STAGING}`.INFORMATION_SCHEMA.COLUMNS
    WHERE table_name IN ('stg_toctoc', 'stg_portal_inmobiliario') AND column_name = 'precio_clp'
''')
display(df)

C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,table_name,column_name,data_type
0,stg_portal_inmobiliario,precio_clp,INT64
1,stg_toctoc,precio_clp,INT64


## 9. Validación: no se eliminaron filas por el tratamiento de superficie

In [15]:
for tabla_raw, tabla_stg in [("toctoc", "stg_toctoc"), ("portal_inmobiliario", "stg_portal_inmobiliario")]:
    ids_distintos = query(f'''
        SELECT COUNT(DISTINCT id) AS n FROM `{PROJECT}.{DATASET_RAW}.{tabla_raw}`
    ''')["n"][0]
    filas_staging = query(f'''
        SELECT COUNT(*) AS n FROM `{PROJECT}.{DATASET_STAGING}.{tabla_stg}`
    ''')["n"][0]
    print(f"{tabla_stg}: {filas_staging} filas vs. {ids_distintos} ids distintos en raw "
          f"→ coincide: {filas_staging == ids_distintos} "
          f"(solo se perdieron duplicados exactos ya conocidos, ninguna fila por superficie)")

C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


stg_toctoc: 124202 filas vs. 124202 ids distintos en raw → coincide: True (solo se perdieron duplicados exactos ya conocidos, ninguna fila por superficie)


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


stg_portal_inmobiliario: 122095 filas vs. 122095 ids distintos en raw → coincide: True (solo se perdieron duplicados exactos ya conocidos, ninguna fila por superficie)


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## Resumen de validación y decisiones

### Hallazgos

- `superficie_m2`/`superficie_total_m2` tiene 8 valores exactos (1, 2, 5, 250, 400, 650, 999,
  1200) que aparecen como picos aislados sin ningún valor vecino, repartidos parejo en los 24
  meses de ambas fuentes (~0.87-0.90% del total cada una), siempre en `tipo_propiedad =
  departamento`.
- `999`/`1200` correlacionan fuertemente con precio extremo (56% Toctoc, 97-99% Portal).
- `250`/`400`/`650` comparten la firma estadística pero **no** el comportamiento de precio: su
  `precio_por_m2` es más bajo que la mediana, compatible con propiedades grandes reales.
- `1`/`2`/`5` son el origen directo de los `precio_por_m2` más extremos del dataset, pero el
  precio absoluto de esas filas es normal — la anomalía está en la superficie.
- El precio máximo de PortalInmobiliario ($141.035.000) tiene `superficie_total_m2 = 1200`, ya
  explicado por el patrón de superficie centinela — no es una anomalía de precio independiente.
- Los duplicados exactos (172 Toctoc / 178 Portal) son copias literales dentro de un mismo
  archivo mensual, sin conflicto de contenido entre filas del mismo grupo.

### Decisiones

- Tratar los 8 valores (1, 2, 5, 250, 400, 650, 999, 1200) como centinela para efectos de
  métricas de superficie, con distinto nivel de evidencia documentado: 999/1200 con evidencia
  fuerte, 250/400/650 por decisión metodológica (no error demostrado), 1/2/5 por generar
  precio/m² artificialmente extremos.
- No eliminar ni modificar filas ni precios por este tratamiento — solo `superficie_m2` pasa a
  `NULL`, conservando `superficie_m2_original` y `superficie_es_centinela` para trazabilidad.
- No definir un umbral arbitrario para excluir precios extremos.
- `precio_clp` se castea a `INT64` (pesos chilenos sin decimales).
- Mantener `precio_uf` nulo en PortalInmobiliario sin imputar con la UF oficial.
- Completar la moneda faltante en Toctoc usando la UF oficial de `fecha_scraping`, con
  trazabilidad vía `precio_clp_es_calculado` / `precio_uf_es_calculado` / `valor_uf_referencia`.
- Deduplicar por fila exacta (`SELECT DISTINCT`) en staging.

### Validaciones

- 0 filas con `superficie_es_centinela = TRUE` y `superficie_m2` no nulo (sección 7).
- 0 filas con `superficie_es_centinela = TRUE` y `precio_clp` nulo — el precio nunca se pierde
  (sección 7).
- Caso `PI-2024-12-003280` verificado individualmente: precio intacto, superficie en `NULL`,
  original conservado (sección 7).
- `precio_clp` confirmado como `INT64` en ambas tablas de staging (sección 8).
- Filas de staging = ids distintos en raw en ambas fuentes → ninguna fila fue eliminada por el
  tratamiento de superficie, solo por la deduplicación exacta ya aprobada (sección 9).